# 3.  Star Catalogues - Under construction!

In this tutorial we show how you can create a custom star catalogue. Later we show how to create a synthetic star catalogue. We note that realistic star catalogues can be generated as part of the PLATOnium toolkit (which another notebook is dedicated for).

### Setup notebook

In [ ]:
# Alow changes to the PlatoSim code outside this notebook
%load_ext autoreload
%autoreload 2

# To interact with the plot use
%matplotlib notebook

### Imports

In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

# PlatoSim
import platosim.referenceFrames as rf
from platosim.utilities    import getFunctions
from platosim.simulation   import Simulation
from platosim.simfile      import SimFile
from platosim.matplotlibrc import setup_notebook
setup_notebook()

In [ ]:
# Inputs
# inputDir  = os.getenv("PLATO_PROJECT_HOME") + "/inputfiles"
# inputFile = inputDir + "/inputfile.yaml"
outputDir = os.getcwd()

---
## 3.1 - Generate a *custom* star catalog
---

As a first example we show how to create a small custom stellar catalogue prior to running PlatoSim. As always we first define the inputs and outputs:

In [ ]:
# Set up a Simulation object
sim = Simulation("output_example1", outputDir=outputDir)

You can also cutomize the orientation (pointing and rotation) of the spacecraft before defining your stellar catalogue:

In [ ]:
sim["ObservingParameters/RApointing"]   = 180.0 
sim["ObservingParameters/DecPointing"]  = -70.0
sim["Platform/SolarPanelOrientation"]   = 0.0

We then set a subfield around the stars being large enough to contain all of them ($50\times50$ pixels):

In [ ]:
sim["SubField/ZeroPointRow"]    = 3000
sim["SubField/ZeroPointColumn"] = 3000
sim["SubField/NumColumns"]      = 50
sim["SubField/NumRows"]         = 50

Specify the pixel coordinates (of the CCD, not of the subfield) of your stars, and create the star catalog file: an ascii file will be written with the columns: `ra`, `dec`, and `mag`. We will use the above defined CCD location to set the rows and columns of the stars:

In [ ]:
# Define catalogue
row = np.array([ 7.0, 40.0, 20.0]) + sim["SubField/ZeroPointRow"]
col = np.array([30.0, 45.0,  5.0]) + sim["SubField/ZeroPointColumn"]
mag = np.array([11.0, 10.0, 12.0])
ID  = [0, 1, 2]

# Automatic catalogue file creation
starcatFile = outputDir + "/starcat_example1.txt"
sim.createStarCatalogFileFromPixelCoordinates(row, col, mag, ID, starcatFile)
row, col

Now, we need to tell PlatoSim to use this newly created catalogue:

In [ ]:
# Make sure the simulation object uses this star catalog
sim["ObservingParameters/StarCatalogFile"] = starcatFile

Finally let's run the simulation and show the stellar pixel map:

In [ ]:
simfile = sim.run(removeOutputFile=True)

In [ ]:
fig, ax = simfile.showImage(clipPercentile=1, imgScale="clip", fontSize=20, 
                            colorMap="cubehelix", colorBar=True, showGrid=True);

---
## 3.2 - Generate a *synthetic* star catalog
---

With the above definitions we can start simulting a realistic synthetic stellar catalogue. For this purpose we have created a `Distribution` class which, among others, cn be used to generate synthetic stellar catalogues. Simple load it with:

In [ ]:
from platosim.distribution import Distribution
filename = os.getenv("PLATO_PROJECT_HOME") + "/inputfiles/starcatalog.txt"
dist = Distribution(filename)

Let's show a little (pretty) table of class functions:

In [ ]:
getFunctions(Distribution)

First we can use `getStarCatalog()` to generate a random catalogue:

In [ ]:
# Star catalogue
row, col, mag, ID = dist.getStarCatalog(numStars=50, minMag=8, maxMag=12.5, 
                                        subfieldNumRows=sim["SubField/NumRows"],
                                        subfieldNumCols=sim["SubField/NumColumns"], 
                                        subfieldZeroRow=sim["SubField/ZeroPointRow"],
                                        subfieldZeroCol=sim["SubField/ZeroPointColumn"],
                                        plot=True)

In [ ]:
# Automatic catalogue file creation
starcatFile = outputDir + "/starcat_example2.txt"
sim.createStarCatalogFileFromPixelCoordinates(row, col, mag, ID, starcatFile)

In [ ]:
# Make sure the simulation object uses this star catalog
sim["ObservingParameters/StarCatalogFile"] = starcatFile

In [ ]:
# Run simulation
simfile = sim.run(removeOutputFile=True)

In [ ]:
# Plot the subfield
fig, ax = simfile.showImage(imageNr=False, clipPercentile=1, imgScale="clip",  
                            figsize=(8,8), fontSize=20, useTitle=False,
                            showStarPositions=False, showStarIDs=False,
                            colorMap="cubehelix", colorBar=True, showGrid=True);

---
## 3.3 - Multi-subfield *synthetic* catalogue creation
---

In the following we show how to create a stellar catalogue, place the stars on different part of the CCD detectors, and simulate each subfield independently over a loop. The importance of such a simulation is due to the fact that PSF increase significantly in size the further away from the optical axis a star is. Let's first define some parameters: 

In [ ]:
# Change these parameters after use
numExposures = 1
numStars     = 10
subfieldDim  = 50
ccdCode      = '4'

We here define five subfields with the following locations:

In [ ]:
# Sub-field related configuration parameters
# (1) "lowerLeft":  lower left  corner of CCD (i.e. close to the readout register, small PSF)
# (2) "lowerRight": lower right corner of CCD (i.e. close to the readout register, large PSF)
# (3) "upperLeft":  upper left  corner of CCD (i.e. far from the readout register, large PSF)
# (4) "upperRight": upper right corner of CCD (i.e. far from the readout register, large PSF)
# (5) "middle":     centre of the CCD (sub-field overlaps with both detector halves)
subfieldNames = ["output_example3_lowerLeft", "output_example3_lowerRight", 
                 "output_example3_upperLeft", "output_example3_upperRight",
                 "output_example3_middle"]
numSubfields  = len(subFieldNames)

# Set subfield dimentions
ccdDimensions         = sim["CCDPositions/NumColumns"][0]
subfieldCenterRows    = [subfieldDim/2, subfieldDim/2, ccdDimensions - subfieldDim/2, 
                         ccdDimensions - subfieldDim/2, ccdDimensions/2]
subfieldCenterColumns = [subfieldDim/2, ccdDimensions - subfieldDim/2, subfieldDim/2, 
                         ccdDimensions - subfieldDim/2, ccdDimensions/2]

Below we use some fix functionalities to place our random stellar catalogue within the subfield for each part of the detector we want to simulate. We run each subfield simulation in a loop as shown here:

In [ ]:
# Run each subfield over a lopp
for i in range(numSubfields):
    
    # Initialise PlatoSim
    sim = Simulation(subfieldNames[i], outputDir=outputDir)
    
    # Set several parameters   
    sim["ObservingParameters/NumExposures"] = numExposures
    sim["CCD/Position"] = str(ccdCode)
    
    # Sky coordinates of subfield: (x, y) = (column, row) -> (RA, Dec) [radians]
    subfieldCenterRa, subfieldCenterDec = rf.pixelToSkyCoordinates(sim, ccdCode, 
                                                                   subfieldCenterColumns[i], 
                                                                   subfieldCenterRows[i])
    
    # Try to set the subfield around the pixel coordinates 
    sim.setSubfieldAroundPixelCoordinates(ccdCode, 
                                          subfieldCenterColumns[i], subfieldCenterRows[i], 
                                          subfieldDim, subfieldDim)

    # Generate star catalogue
    starcatFileName = outputFileName +  subfieldNames[i] + ".txt"
    ra, dec = np.zeros((numStars)), np.zeros((numStars))  
    
    for starIndex in range(numStars):
        
        # Generate random stellar catalogue
        row, col, mag, ID = dist.getStarCatalog(numStars, minMag=8, maxMag=12.5, 
                                                subfieldNumRows=subfieldDim,
                                                subfieldNumCols=subfieldDim, 
                                                subfieldZeroRow=sim["SubField/ZeroPointRow"],
                                                subfieldZeroCol=sim["SubField/ZeroPointColumn"],
                                                plot=False)
        
        # Get the subfield row and column 
        starRow = row[starIndex] + subfieldCenterRows[i]    #- subfieldDim / 2
        starCol = col[starIndex] + subfieldCenterColumns[i] #- subfieldDim / 2
        
        # (x, y) -> (RA, Dec) [radians]
        starRA, starDec = rf.pixelToSkyCoordinates(sim, ccdCode, starCol, starRow)
        ra[i]  = starRA
        dec[i] = starDec
    
    
    # RA should be in range [-PI, PI] and convert to radians
    ra[ra > np.pi] = ra[ra > np.pi] - 2*np.pi
    ra  = np.rad2deg(ra)
    dec = np.rad2deg(dec)
    
    # Store the catalogue
    np.savetxt(starcatFileName, np.transpose([ra, dec, mag]), 
               fmt=['%11.6f', '%11.6f', '%8.4f'])
    sim["ObservingParameters/StarCatalogFile"] = starcatFileName
    
    # Write YAML file 
    configurationFileName = outputFileName + subfieldNames[i] + ".yaml"
    sim.writeYamlConfigurationFile(configurationFileName)
    
    # Run each simulation
    print(subfieldNames[i])
    simfile = sim.run(removeOutputFile=True)

In [ ]:
# Plot the subfield
f = SimFile(subfieldNames[-3] + ".hdf5")
fig, ax = f.showImage(imageNr=False, clipPercentile=1, imgScale="clip", 
                      figsize=(8,8), fontSize=20, useTitle=False,
                      showStarPositions=False, showStarIDs=False,
                      colorMap="cubehelix", colorBar=True, showGrid=True);

---
## 3.4 - Specify a star catalog from pixel coordinates
---

Instead of generating a random star catalogue, it is also possible to place one or more stars that you specify in CCD coordinates instead of equatorial coordinates (RA, Dec). This is especially useful when you want to simulate features at a specific location on the CCD or if you want to supply a grid of stars equally distributed over the PLATO CCDs.

<img src='../../figures/CoordinatesTransformations.png'>

Methods that we will focus on are imported from `referenceFrames.py`.

* pixelToSkyCoordinates
* skyToPixelCoordinates

These are convenience methods that combine the above steps in one simple function and take field distortion into account.

First we create a new simulation object:

In [ ]:
# Set up a Simulation object
sim = Simulation("output_example1", outputDir=outputDir)

In [ ]:
sim["Camera/IncludeFieldDistortion"] = "no"
sim["ObservingParameters/RApointing"] = 10.0
sim["ObservingParameters/DecPointing"] = 10.0
sim["Camera/FocalPlaneOrientation/ConstantValue"] = 0.0

In [ ]:
ccdCode = '2'
xCCDpixel = 255 
yCCDpixel = 255
subfieldSizeX = 50
subfieldSizeY = 50

In [ ]:
import math
import numpy as np
raCenter, decCenter = pixelToSkyCoordinates(sim, ccdCode, xCCDpixel, yCCDpixel)
print(xCCDpixel,  yCCDpixel)

print(math.degrees(raCenter), math.degrees(decCenter))
code, x, y = skyToPixelCoordinates(sim, raCenter, decCenter, True)
print(code, x, y)

In [ ]:
sim.setSubfieldAroundCoordinates(raCenter, decCenter, subfieldSizeX, subfieldSizeY, normal=True)

Make a starCatalog with stars of identical magnitude lying over the diagonals of each CCD

In [ ]:
starCatalogFilename = outputDir + "/starcat_example4.txt"

In [ ]:
vMag = []
raStar = []
decStar = []

for x in range(255, 4510, 200):
    
    ra, dec = pixelToSkyCoordinates(sim, ccdCode, x, x)
    
    raStar.append(ra)
    decStar.append(dec)
    vMag.append(12.5)
    
for x in range(255, 4510, 200):
    
    ra, dec = pixelToSkyCoordinates(sim, ccdCode, x, 4510 - x)
    
    raStar.append(ra)
    decStar.append(dec)
    vMag.append(12.5)

raStar = np.array(raStar)
decStar = np.array(decStar)
vMag = np.array(vMag)

raStar[raStar > math.pi] = raStar[raStar > math.pi] - 2 * math.pi

raStar = np.rad2deg(raStar)
decStar = np.rad2deg(decStar)

In [ ]:
np.savetxt(starCatalogFilename, np.transpose([raStar, decStar, vMag]), fmt=['%11.6f', '%11.6f', '%8.4f'])

Update the simulation parameters to pick up the new star catalog and set the subField around one of the stars.

In [ ]:
sim["ObservingParameters/StarCatalogFile"] = starCatalogFilename

In [ ]:
sim.setSubfieldAroundCoordinates(raCenter, decCenter, subfieldSizeX, subfieldSizeY, normal=True)

In [ ]:
sim.setSubfieldAroundPixelCoordinates(ccdCode, 255, 255, subfieldSizeX, subfieldSizeY)

In [ ]:
RA_PLATFORM  = np.deg2rad(float(sim["ObservingParameters/RApointing"]))
DEC_PLATFORM = np.deg2rad(float(sim["ObservingParameters/DecPointing"]))

In [ ]:
azimuth = np.deg2rad(float(sim["Telescope/AzimuthAngle"]))
tilt    = np.deg2rad(float(sim["Telescope/TiltAngle"]))

In [ ]:
solarPanelOrientation = np.deg2rad(float(sim["Platform/SolarPanelOrientation"]))

In [ ]:
raSun, decSun = sunSkyCoordinatesAwayfromPlatformPointing(RA_PLATFORM, DEC_PLATFORM, solarPanelOrientation)
RA_TELESCOPE, DEC_TELESCOPE = platformToTelescopePointingCoordinates(RA_PLATFORM, DEC_PLATFORM, raSun, decSun, azimuth, tilt)

In [ ]:
focalPlaneAngle = np.deg2rad(float(sim["Camera/FocalPlaneOrientation/ConstantValue"]))       # [rad]
focalLength     = float(sim["Camera/FocalLength/ConstantValue"])*1000.                       # [m] -> [mm]
pixelSize       = float(sim["CCD/PixelSize"])                                                # [micron]

#### Check the result by plotting the CCDs and the stars on the sky in a aitprojection in equatorial coordinates:

In [ ]:
fig = plt.figure()
axes = drawCCDsInSkyMollweide(fig, RA_PLATFORM, DEC_PLATFORM, solarPanelOrientation, tilt, azimuth, focalPlaneAngle, focalLength, pixelSize)
axes.scatter(-np.deg2rad(raStar), np.deg2rad(decStar))

#### Plot the CCDs and the stars from the star catalog in the focal plane.

In [ ]:
ra, dec, Vmag = np.loadtxt(starCatalogFilename, unpack=True)
ra = np.deg2rad(ra)
dec = np.deg2rad(dec)

Nstars = len(ra)

The optical axis in the plot above points towards the reader.

In [ ]:
# Draw the CCD squares, with the readout register as a thicker line

plot.drawCCDsInFocalPlane(pixelSize, plotCCDlabels=True, normal=True)

# Draw the stars on the CCD

for idx in range(0, Nstars):
    plot.drawStarInFocalPlane(sim, ra[idx], dec[idx])

# Plot the X axis of the telescope reference frame in this focal plane

(xTL, yTL, zTL) = (0.5, 0.0, np.sqrt(1-0.5**2))             # unit vector
xFP, yFP = referenceFrames.telescopeToUndistortedFocalPlaneCoordinates(xTL, yTL, zTL, focalLength, focalPlaneAngle)
length = np.sqrt(xFP**2+yFP**2)
xFP *= 85.0 / length
yFP *= 85.0 / length
plt.arrow(0.0, 0.0, xFP, yFP, head_width=3, color="k")
plt.text(xFP-1.5, yFP+4, "xTL", fontsize=12, color="k")

# Plot the Y axis of the telescope reference frame in this focal plane

(xTL, yTL, zTL) = (0.0, 0.5, np.sqrt(1-0.5**2))             # unit vector
xFP, yFP = referenceFrames.telescopeToUndistortedFocalPlaneCoordinates(xTL, yTL, zTL, focalLength, focalPlaneAngle)
length = np.sqrt(xFP**2+yFP**2)
xFP *= 85.0 / length
yFP *= 85.0 / length
plt.arrow(0.0, 0.0, xFP, yFP, head_width=3, color="k")
plt.text(xFP-9, yFP, "yTL", fontsize=12, color="k")

plt.axis('scaled')